In [97]:
import re

In [98]:
src = open("ex1.py", 'r').read()
src

'import sys\nsys.path.insert(0, "../") # noqa\nimport sys\nfrom n3.parse import parse_n3_file\nfrom n3.objects import ANY, Terms, Iri, Var, Literal, Collection, GraphTerm, Triple\nfrom n3.ns import NS\nfrom lib.emit import emit, emitted\n\ndata = parse_n3_file(\'/Users/wvw/git/n3/fun3/python/tests-bench/zika/data/gen100_pt2.n3\').data\n\n# rule_0 (id_1=x_0) > rule_1 (p_3=ANY) [RULE - ret: p_3_m=? ... s_2]\n#   > (exec) data.find(p_3,rdf:type,fhir:Patient) (ret: s_1,p_1,o_1) > rule_1_1 (p_4=s_1) > (exec) data.find(p_4,fhir:hasCondition,ANY) (s_2,p_2,o_2; FINAL=s_2)\n# > rule_0_1 (id_1=id_1,p_2=p_3_m) > (exec) data.find(p_2,fhir:id,id_1) (ret: s_3,p_3,o_3) > FINAL=o_3\n\n# for s_1,p_1,o_1 in data.find(ANY,rdf:type,fhir:Patient):\n#   for s_2, p_2, o_2 in data.find(s_1,fhir:hasCondition,ANY):\n#       for s_3, p_3, o_3 in data.find(s_2,fhir:id,x_0):\n#           yield o_3\n\ndef query(x_0, final_ctu):\n    rule_0(x_0, lambda id_1_m: final_ctu(id_1_m))\n\ndef rule_0(id_1, final_ctu):\n    

In [99]:
from ast import dump, parse

mod = parse(src)
print(dump(mod, indent=2))

Module(
  body=[
    Import(
      names=[
        alias(name='sys')]),
    Expr(
      value=Call(
        func=Attribute(
          value=Attribute(
            value=Name(id='sys', ctx=Load()),
            attr='path',
            ctx=Load()),
          attr='insert',
          ctx=Load()),
        args=[
          Constant(value=0),
          Constant(value='../')])),
    Import(
      names=[
        alias(name='sys')]),
    ImportFrom(
      module='n3.parse',
      names=[
        alias(name='parse_n3_file')],
      level=0),
    ImportFrom(
      module='n3.objects',
      names=[
        alias(name='ANY'),
        alias(name='Terms'),
        alias(name='Iri'),
        alias(name='Var'),
        alias(name='Literal'),
        alias(name='Collection'),
        alias(name='GraphTerm'),
        alias(name='Triple')],
      level=0),
    ImportFrom(
      module='n3.ns',
      names=[
        alias(name='NS')],
      level=0),
    ImportFrom(
      module='lib.emit',
      names=[

In [100]:
from ast import FunctionDef, Name

# TODO
# support for if conditions (unification)

name_fndef = { el.name: el for el in mod.body if isinstance(el, FunctionDef) }

def nodestr(node):
    if isinstance(node, Name):
        return node.id
    elif isinstance(node, str):
        return node
    else:
        return dump(node)
    
def varmapstr(var_map):
    return { param: nodestr(arg) for param, arg in var_map.items() }

def visit(fn_name, var_map):
    fn_def = name_fndef[fn_name]
    fn_params = [ param.arg for param in fn_def.args.args ][:-1]
    
    for stmt in fn_def.body:
        call_var_map = var_map.copy()
        fn_call = stmt.value
        
        lmbda_def = fn_call.args[-1]
        # print(dump(lmbda_def, indent=4))        
        lmbda_params = [ param.arg for param in lmbda_def.args.args ]
        
        ctu_fn_name = lmbda_def.body.func.id
        ctu_fn_args = [ arg.id for arg in lmbda_def.body.args ]
        
        if isinstance(fn_call.func, Name): # rule invocation
            called_fn_name = fn_call.func.id
            called_fn = name_fndef[called_fn_name]
            called_fn_params = [ param.arg for param in called_fn.args.args ][:-1]
            
            # map the provided args to the rule fn's params
            for param, arg in zip(called_fn_params, fn_call.args):
                call_var_map[param] = arg
            print(fn_name, "->", called_fn_name, varmapstr(call_var_map))
            
            # this will return final_ctu's args; i.e., what the new rule fn will yield
            callback_args = visit(called_fn_name, call_var_map)
            for param, arg in zip(lmbda_params, callback_args):
                call_var_map[param] = arg
            print("ret", called_fn_name, varmapstr(call_var_map))
        
        # this is what the rule fn will eventually return        
        if ctu_fn_name == 'final_ctu':
            return ctu_fn_args
        else:
            # else, follow the compile-time thread to the final_ctu call
            # next internal ctu
            ctu_fn = name_fndef[ctu_fn_name]
            # params of that int ctu fn
            ctu_fn_params = [ param.arg for param in ctu_fn.args.args ][:-1]
            
            for param, arg in zip(ctu_fn_params, ctu_fn_args):
                call_var_map[param] = arg
            print(ctu_fn_name, varmapstr(call_var_map))
            
            return visit(ctu_fn_name, call_var_map)
            

query_node = name_fndef['query']
print(dump(query_node, indent=2))

var_map = {}
visit('query', var_map)

FunctionDef(
  name='query',
  args=arguments(
    args=[
      arg(arg='x_0'),
      arg(arg='final_ctu')]),
  body=[
    Expr(
      value=Call(
        func=Name(id='rule_0', ctx=Load()),
        args=[
          Name(id='x_0', ctx=Load()),
          Lambda(
            args=arguments(
              args=[
                arg(arg='id_1_m')]),
            body=Call(
              func=Name(id='final_ctu', ctx=Load()),
              args=[
                Name(id='id_1_m', ctx=Load())]))]))])
query -> rule_0 {'id_1': 'x_0'}
rule_0 -> rule_1 {'id_1': 'x_0', 'p_3': 'ANY'}
rule_1_1 {'id_1': 'x_0', 'p_3': 'ANY', 'p_4': 's3'}
ret rule_1 {'id_1': 'x_0', 'p_3': 'ANY', 'p_3_m': 's4'}
rule_0_1 {'id_1': 'id_1', 'p_3': 'ANY', 'p_3_m': 's4', 'p_2': 'p_3_m'}
ret rule_0 {'id_1': 'x_0', 'id_1_m': 'o2'}


['id_1_m']